# 🚀 Dijkstra's Algorithm — An Interactive Walkthrough

*Single-Source Shortest Paths on graphs with **non-negative** edge weights.*

**Course:** Algorithms (EE4033) · Lecture 11 — Shao-Hua Sun, NTU EE

---

Given a weighted graph $G = (V, E)$ and a **source** vertex $s$, Dijkstra's
algorithm finds the shortest-path distance $\delta(s, v)$ from $s$ to **every**
other vertex $v$ — provided all edge weights are non-negative.

## What you will learn

By the end of this notebook you will be able to:

1. **Explain** the greedy idea: grow a set $S$ of *finalized* vertices, always
   finalizing the closest remaining vertex.
2. **Trace** the algorithm step-by-step on the classic textbook graph and watch
   distance estimates get *relaxed*.
3. **Implement** Dijkstra with a binary-heap priority queue in a few lines.
4. **Predict** the running time for array vs. heap priority queues.
5. **See why non-negative weights matter** — and watch the algorithm fail when
   they don't.

> ▶️ Run the cells top-to-bottom (**Runtime → Run all**). Cells marked 🎛️ are
> interactive — drag the sliders!

## 1. The idea in one picture

For every vertex $v$, Dijkstra keeps track of two values:

| Symbol | Meaning |
|---|---|
| $v.d$ | current **shortest-path estimate** — length of the best $s\to v$ path found *so far* (starts at $\infty$, with $s.d = 0$) |
| $v.\pi$ | the **predecessor** of $v$ on that best path (lets us rebuild the route) |

It keeps a set $S$ of vertices whose distance is already **final**, and
repeatedly performs this **greedy step**:

> Among all vertices *not yet finalized*, pick the one $u$ with the smallest
> estimate $u.d$, declare it final ($S \leftarrow S \cup \{u\}$), and **relax**
> every edge leaving $u$.

**Relaxing** an edge $(u, v)$ asks: *"is going through $u$ a shortcut to $v$?"*

$$\textbf{Relax}(u,v,w):\quad \text{if } v.d > u.d + w(u,v)\ \text{ then }\ v.d \leftarrow u.d + w(u,v),\ \ v.\pi \leftarrow u$$

### Pseudocode (from the lecture)

```text
Dijkstra(G, w, s)
1  Initialize-Single-Source(G, s)   # all v.d = inf, s.d = 0
2  S = empty
3  Q = G.V                          # min-priority queue keyed by v.d
4  while Q is not empty
5      u = Extract-Min(Q)           # greedy choice: nearest vertex
6      S = S + {u}
7      for each v in G.Adj[u]
8          Relax(u, v, w)
```

It is a **greedy** algorithm and runs *essentially the same as Prim's MST
algorithm* — the difference is only the key used to choose the next vertex.

In [ ]:
# Setup: imports, optional widgets, and consistent plot styling.
import heapq
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle

# ipywidgets is optional. Guard the import so "Run all" never aborts on a
# runtime without widgets; the 3 interactive cells fall back to static plots.
try:
    from ipywidgets import interact, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("WARNING: ipywidgets not available -> interactive cells render statically.")

INF = float("inf")

# Consistent styling across the whole notebook.
plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "lines.linewidth": 2,
})

# Sanity checks
assert callable(plt.plot) and hasattr(np, "log2")
print("ipywidgets available:", WIDGETS_AVAILABLE)

## 2. Meet the example graph

We use the **same directed graph as the lecture slides** — vertices
$\{s, t, x, y, z\}$ with the source at $s$. Each arrow is a one-way edge labeled
with its (non-negative) weight.

**Color legend for every drawing below:**

- ⚫ **black** node = *finalized*, already in $S$ (its distance is optimal)
- 🟡 **gold** node = the *current* vertex $u$ just extracted from the queue
- ⚪ **white** node = still waiting in the queue $Q$
- 🔴 **crimson** arrow = an edge of the shortest-path tree (a $v.\pi$ pointer)
- 🟢 **green** arrow = an edge that was *just relaxed* in this step

The number inside each node is its current estimate $v.d$ ($\infty$ = not
reached yet).

In [ ]:
# The example graph as an adjacency list:  vertex -> [(neighbor, weight), ...]
GRAPH = {
    "s": [("t", 10), ("y", 5)],
    "t": [("x", 1), ("y", 2)],
    "x": [("z", 4)],
    "y": [("t", 3), ("x", 9), ("z", 2)],
    "z": [("x", 6), ("s", 7)],
}
VERTICES = ["s", "t", "x", "y", "z"]   # fixed order for stable tables

# Fixed 2-D positions, matching the slide layout (s left, t/x top, y/z bottom).
POS = {
    "s": (0.0, 1.0),
    "t": (1.6, 2.0),
    "x": (3.2, 2.0),
    "y": (1.6, 0.0),
    "z": (3.2, 0.0),
}

NODE_R = 0.27  # node-circle radius in data coordinates


def draw_graph(dist=None, pred=None, S=None, u=None, relaxed=None,
               title="", ax=None, graph=GRAPH, pos=POS):
    """Draw the graph, coloring vertices by Dijkstra status and highlighting
    the shortest-path (predecessor) tree.

    dist    : {vertex: estimate}     -> shown inside each node
    pred    : {vertex: predecessor}  -> crimson tree edges
    S       : iterable of finalized vertices (black)
    u       : the current vertex being processed (gold; drawn on top of S)
    relaxed : list of (a, b) edges just relaxed (green highlight)
    """
    S = set(S or [])
    pred = pred or {}
    relaxed = set(relaxed or [])
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.6))

    edge_set = {(a, b) for a in graph for b, _ in graph[a]}

    # ---- edges (arrows + weight labels) ----
    for a in graph:
        for b, w in graph[a]:
            curved = (b, a) in edge_set          # bidirectional pair -> curve both
            rad = 0.18 if curved else 0.0
            if (a, b) in relaxed:
                color, lw, z = "tab:green", 3.2, 4
            elif pred.get(b) == a:               # edge belongs to the SP tree
                color, lw, z = "crimson", 3.0, 3
            else:
                color, lw, z = "0.6", 1.4, 1
            ax.add_patch(FancyArrowPatch(
                pos[a], pos[b], connectionstyle="arc3,rad=%g" % rad,
                arrowstyle="-|>", mutation_scale=14, lw=lw, color=color,
                shrinkA=16, shrinkB=16, zorder=z))
            # place the weight label near the (possibly curved) edge
            pa, pb = np.asarray(pos[a], float), np.asarray(pos[b], float)
            mid = 0.5 * (pa + pb)
            d = pb - pa
            length = float(np.hypot(*d))
            perp = np.array([-d[1], d[0]]) / length     # unit, left of a->b
            off = 0.5 * rad * length + 0.16
            lp = mid + perp * off
            ax.text(lp[0], lp[1], str(w), fontsize=10, ha="center",
                    va="center", zorder=5,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white",
                              ec="none", alpha=0.85))

    # ---- nodes ----
    cx = sum(p[0] for p in pos.values()) / len(pos)   # graph centroid: push
    cy = sum(p[1] for p in pos.values()) / len(pos)   # name labels outward
    for v, (x, y) in pos.items():
        if v == u:                 # current vertex wins over S
            fc, tc = "gold", "black"
        elif v in S:
            fc, tc = "black", "white"
        else:
            fc, tc = "white", "black"
        ax.add_patch(Circle((x, y), NODE_R, fc=fc, ec="black", lw=1.8, zorder=6))
        # place the vertex name on the OUTER side (away from the centroid) so it
        # never collides with interior edge-weight labels.
        ox, oy = x - cx, y - cy
        norm = (ox * ox + oy * oy) ** 0.5 or 1.0
        lx = x + ox / norm * (NODE_R + 0.20)
        ly = y + oy / norm * (NODE_R + 0.20)
        ax.text(lx, ly, v, fontsize=12, fontweight="bold", ha="center",
                va="center", zorder=8,
                bbox=dict(boxstyle="round,pad=0.05", fc="white", ec="none",
                          alpha=0.7))
        if dist is not None:
            dv = dist.get(v, INF)
            lab = "∞" if dv == INF else "%g" % dv
            ax.text(x, y, lab, fontsize=12, color=tc, ha="center",
                    va="center", zorder=7)

    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.8, max(xs) + 0.8)
    ax.set_ylim(min(ys) - 0.8, max(ys) + 0.8)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=11)
    return ax


# Show the initial state: s.d = 0, every other estimate is infinity.
init_dist = {v: INF for v in VERTICES}
init_dist["s"] = 0
draw_graph(dist=init_dist, S=set(), title="Initial state:  s.d = 0,  all others = ∞")
plt.show()

In [ ]:
# --- Core Dijkstra: binary-heap (priority-queue) version --------------------
def all_vertices(graph):
    """Every vertex that appears as a key OR as a neighbor."""
    vs = set(graph)
    for a in graph:
        for b, _ in graph[a]:
            vs.add(b)
    return vs


def dijkstra(graph, source):
    """Return (dist, pred) dicts of shortest distances and predecessors."""
    dist = {v: INF for v in all_vertices(graph)}
    pred = {v: None for v in all_vertices(graph)}
    dist[source] = 0
    S = set()                       # finalized vertices
    pq = [(0, source)]              # min-heap of (estimate, vertex)

    while pq:
        d_u, u = heapq.heappop(pq)
        if u in S:                  # stale heap entry -> skip (lazy deletion)
            continue
        S.add(u)                    # u's distance is now FINAL
        for v, w in graph.get(u, []):
            if v not in S and dist[u] + w < dist[v]:   # Relax(u, v, w)
                dist[v] = dist[u] + w
                pred[v] = u
                heapq.heappush(pq, (dist[v], v))
    return dist, pred


def reconstruct_path(pred, source, target):
    """Follow predecessors back from target to source; None if unreachable."""
    if target != source and pred.get(target) is None:
        return None
    path, v = [], target
    while v is not None:
        path.append(v)
        v = pred.get(v)
    path.reverse()
    return path


dist, pred = dijkstra(GRAPH, "s")

print("Shortest distances from source 's':\n")
print(f"{'vertex':>7}{'distance':>10}{'predecessor':>13}   path")
print("-" * 45)
for v in VERTICES:
    path = reconstruct_path(pred, "s", v)
    print(f"{v:>7}{dist[v]:>10g}{str(pred[v] or '-'):>13}   {' -> '.join(path)}")

# Sanity check against the worked example on the lecture slides.
expected = {"s": 0, "t": 8, "x": 9, "y": 5, "z": 7}
assert dist == expected, f"got {dist}, expected {expected}"
print("\n✓ Matches the lecture's worked example:", expected)

## 3. Watch it run, one step at a time 🎛️

The clearest way to understand Dijkstra is to watch $S$ grow. Below we record a
**snapshot after every `Extract-Min`** and let you scrub through them.

At each step, read it as: *"we just pulled the closest white node out of the
queue, painted it gold, finalized it, and relaxed its outgoing edges (green)."*

Drag the **`step`** slider from `0` (initialization) to the end (all vertices
finalized) and compare with the distance table on the right.

In [ ]:
# Instrumented Dijkstra: record the full state after each Extract-Min so we
# can replay the algorithm frame by frame.
def dijkstra_steps(graph, source):
    dist = {v: INF for v in all_vertices(graph)}
    pred = {v: None for v in all_vertices(graph)}
    dist[source] = 0
    S, pq, steps = set(), [(0, source)], []

    # step 0: initialization
    steps.append(dict(u=None, S=set(), dist=dict(dist), pred=dict(pred),
                      relaxed=[], message=f"Initialize: {source}.d = 0, "
                                          f"every other d = infinity."))
    while pq:
        d_u, u = heapq.heappop(pq)
        if u in S:
            continue
        S.add(u)
        relaxed = []
        for v, w in graph.get(u, []):
            if v not in S and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u
                heapq.heappush(pq, (dist[v], v))
                relaxed.append((u, v))
        msg = (f"Extract-Min -> u = '{u}' (d = {d_u:g}); add to S. "
               + (f"Relaxed: " + ", ".join(f"{a}->{b}" for a, b in relaxed)
                  if relaxed else "No edge improved."))
        steps.append(dict(u=u, S=set(S), dist=dict(dist), pred=dict(pred),
                          relaxed=relaxed, message=msg))
    return steps


STEPS = dijkstra_steps(GRAPH, "s")
print(f"Recorded {len(STEPS)} steps (step 0 = init, then one per finalized vertex).")

In [ ]:
# 🎛️ Interactive step scrubber: graph on the left, state table on the right.
def show_step(i):
    step = STEPS[i]
    fig, (axg, axt) = plt.subplots(
        1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [3, 2]})
    draw_graph(dist=step["dist"], pred=step["pred"], S=step["S"],
               u=step["u"], relaxed=step["relaxed"],
               title=f"Step {i} / {len(STEPS) - 1}", ax=axg)

    axt.axis("off")
    lines = [f"{'node':<6}{'d':>5}{'pi':>5}{'status':>11}", "-" * 27]
    for v in VERTICES:
        dv = step["dist"][v]
        ds = "inf" if dv == INF else f"{dv:g}"
        pi = step["pred"][v] or "-"
        if v == step["u"]:
            st = "current"
        elif v in step["S"]:
            st = "S (final)"
        else:
            st = "Q"
        lines.append(f"{v:<6}{ds:>5}{pi:>5}{st:>11}")
    axt.text(0.0, 1.0, "\n".join(lines), family="monospace", fontsize=12,
             va="top", ha="left", transform=axt.transAxes)
    axt.text(0.0, 0.22, step["message"], fontsize=10, color="navy",
             va="top", ha="left", wrap=True, transform=axt.transAxes)
    plt.tight_layout()
    plt.show()


if WIDGETS_AVAILABLE:
    interact(show_step,
             i=IntSlider(min=0, max=len(STEPS) - 1, step=1, value=0,
                         description="step", continuous_update=False))
else:
    for i in range(len(STEPS)):     # static fallback: render every frame
        show_step(i)

## 4. Change the source vertex 🎛️

The shortest-path *tree* depends on where you start. Pick a different **source**
below and watch the crimson tree and the path list update. (All edges are
non-negative here, so Dijkstra is guaranteed correct.)

In [ ]:
# 🎛️ Recompute and draw the shortest-path tree from any chosen source.
def explore_source(source):
    dist_s, pred_s = dijkstra(GRAPH, source)
    draw_graph(dist=dist_s, pred=pred_s, S=set(VERTICES), u=source,
               title=f"Shortest-path tree from source '{source}'")
    plt.show()
    print(f"Shortest paths from '{source}':")
    for v in VERTICES:
        path = reconstruct_path(pred_s, source, v)
        if dist_s[v] == INF:
            print(f"  {source} -> {v}: unreachable")
        else:
            print(f"  {source} -> {v}: d = {dist_s[v]:<3g} via {' -> '.join(path)}")


if WIDGETS_AVAILABLE:
    interact(explore_source,
             source=Dropdown(options=VERTICES, value="s", description="source"))
else:
    explore_source("s")

## 5. Why "non-negative" is not optional ⚠️

Dijkstra's correctness rests on one fact: **the moment a vertex is extracted,
its estimate is already optimal.** That is only guaranteed when every edge
weight is $\ge 0$ — adding more edges can never make a path *shorter*, so a
finalized vertex can never be improved later.

Break that assumption and the greedy choice can finalize a vertex **too early**.
Consider this tiny graph with one negative edge:

```text
   s --2--> a
   s --3--> b
   b -(-2)-> a      <-- negative!
```

The true shortest $s\to a$ path is the *detour* $s\to b\to a = 3 + (-2) = 1$,
which is cheaper than the direct edge of weight $2$. But Dijkstra extracts `a`
(estimate $2$) **before** `b` (estimate $3$), finalizes it, and never looks
back. **Bellman-Ford**, which has no such early-finalization rule, gets it
right.

In [ ]:
# A graph that VIOLATES the non-negative-weight assumption.
NEG_GRAPH = {
    "s": [("a", 2), ("b", 3)],
    "b": [("a", -2)],          # negative edge
    "a": [],
}


def bellman_ford(graph, source):
    """Shortest distances allowing negative edges (no negative cycles)."""
    V = all_vertices(graph)
    dist = {v: INF for v in V}
    dist[source] = 0
    for _ in range(len(V) - 1):                 # |V|-1 relaxation passes
        for x in V:
            for y, w in graph.get(x, []):
                if dist[x] + w < dist[y]:
                    dist[y] = dist[x] + w
    return dist


dij_dist, _ = dijkstra(NEG_GRAPH, "s")
bf_dist = bellman_ford(NEG_GRAPH, "s")

print(f"{'vertex':>7}{'Dijkstra':>10}{'Bellman-Ford':>14}   verdict")
print("-" * 46)
for v in ["s", "a", "b"]:
    verdict = "ok" if dij_dist[v] == bf_dist[v] else "WRONG  <-- finalized too early!"
    print(f"{v:>7}{dij_dist[v]:>10g}{bf_dist[v]:>14g}   {verdict}")

print("\nDijkstra reports dist(s, a) = 2 (the direct edge) but the true answer")
print("is 1, via the cheaper detour s -> b -> a that uses the negative edge.")

In [ ]:
# 🎛️ Slide the weight of edge b->a and see when Dijkstra diverges from truth.
def negative_demo(w_ba):
    g = {"s": [("a", 2), ("b", 3)], "b": [("a", w_ba)], "a": []}
    d_a = dijkstra(g, "s")[0]["a"]          # Dijkstra's estimate for a
    t_a = bellman_ford(g, "s")["a"]         # true shortest distance to a
    agree = d_a == t_a

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(["Dijkstra", "Bellman-Ford\n(true)"], [d_a, t_a],
                  color=[("tab:blue" if agree else "tab:orange"), "tab:green"])
    for rect, val in zip(bars, [d_a, t_a]):
        ax.text(rect.get_x() + rect.get_width() / 2, val,
                f"{val:g}", ha="center", va="bottom")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel("shortest distance to a")
    ax.set_ylim(top=max(d_a, t_a, 2) + 1)
    verdict = "AGREE ✓  (Dijkstra correct)" if agree else "DISAGREE ✗  (Dijkstra WRONG)"
    ax.set_title(f"w(b->a) = {w_ba}   ->   {verdict}")
    plt.show()


if WIDGETS_AVAILABLE:
    interact(negative_demo,
             w_ba=IntSlider(min=-3, max=2, step=1, value=-2,
                            description="w(b->a)", continuous_update=False))
else:
    for wv in (-2, 0):
        negative_demo(wv)
print("Try it: for w(b->a) >= -1 they agree; below -1 the detour wins and")
print("Dijkstra's early finalization of 'a' becomes wrong.")

## 6. How fast is it? ⏱️

The running time depends entirely on how the priority queue $Q$ is implemented:

| $Q$ implementation | Extract-Min | Decrease-Key (relax) | **Total** |
|---|---|---|---|
| linear array | $O(V)$ | $O(1)$ | $O(V^2)$ |
| **binary heap** | $O(\log V)$ | $O(\log V)$ | $O(E \log V)$ |
| Fibonacci heap | $O(\log V)$ amort. | $O(1)$ amort. | $O(E + V\log V)$ |

- The `while` loop runs **Extract-Min** exactly $V$ times.
- Over the whole run, edges are relaxed $O(E)$ times.

For **sparse** graphs ($E = O(V)$) the binary heap wins; for **dense** graphs
($E = O(V^2)$) the plain array is asymptotically just as good. Let's *measure*
the heap version on random graphs of growing size.

In [ ]:
# Empirically time the heap-based Dijkstra and compare to a c * E*log2(V) curve.
def random_graph(n, avg_deg, rng):
    g = {i: [] for i in range(n)}
    for _ in range(avg_deg * n):
        a, b = int(rng.integers(n)), int(rng.integers(n))
        if a != b:
            g[a].append((b, int(rng.integers(1, 100))))   # non-negative weights
    return g


sizes = [50, 100, 200, 400, 800, 1600]
avg_deg = 8
rng = np.random.default_rng(0)
times = []
for n in sizes:
    g = random_graph(n, avg_deg, rng)
    t0 = time.perf_counter()
    for _ in range(3):                      # average a few runs
        dijkstra(g, 0)
    times.append((time.perf_counter() - t0) / 3)

E = np.array([avg_deg * n for n in sizes], float)
elogv = E * np.log2(np.maximum(sizes, 2))
scale = times[-1] / elogv[-1] if elogv[-1] else 1.0

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sizes, np.array(times) * 1e3, "o-", label="measured heap Dijkstra")
ax.plot(sizes, elogv * scale * 1e3, "k--", label="reference  c · E log₂ V")
ax.set_xlabel("number of vertices |V|   (|E| ≈ 8|V|)")
ax.set_ylabel("time per run (ms)")
ax.set_title("Binary-heap Dijkstra scales like O(E log V)")
ax.legend()
plt.show()

## 7. Your turn — questions to think about 🧠

1. **Greedy safety.** Dijkstra finalizes a vertex the instant it is extracted
   and never revisits it. *Why* is that safe when weights are non-negative?
   Could a later, longer path ever re-enter a finalized vertex and improve it?
2. **Relaxation in action.** In the step scrubber, vertex `t` first gets
   estimate $10$ (directly from `s`) and later improves to $8$. Which step makes
   that happen, and which edge causes it?
3. **Ties.** If two vertices share the same estimate at `Extract-Min`, does it
   matter which we pick first? Will the final distances change?
4. **Undirected graphs.** Our graph is directed. How would you represent an
   undirected edge $\{u, v\}$ of weight $w$ so this exact code still works?
5. **The breaking point.** In §5, at *exactly* which value of `w(b→a)` do
   Dijkstra and Bellman-Ford start to disagree? Explain the threshold.
6. **Complexity intuition.** On the timing plot, the measured curve tracks
   $c\cdot E\log V$. For *sparse* graphs, what shape would the array-based
   $O(V^2)$ version trace on the same axes?

## 8. Suggested extensions 🛠️

- **Highlight one route.** Extend `explore_source` to take a *target* and draw
  only the single $s\to t$ shortest path in bold.
- **A\* search.** Add a heuristic $h(v)$ (e.g. straight-line distance to the
  goal) and change the priority to $v.d + h(v)$. On a grid, watch A\* expand far
  fewer nodes than Dijkstra.
- **Early exit.** If you only need the distance to *one* target, stop the loop
  the moment that target is extracted. Argue why that is correct.
- **Generalize to negative edges.** Swap in Bellman-Ford / SPFA and compare
  runtimes on the same graphs.
- **Grid pathfinding.** Build a 2-D grid with obstacles, run Dijkstra, and
  animate the expanding frontier — the classic "flood fill" intuition.

## 9. One-paragraph recap

Dijkstra's algorithm grows a set $S$ of vertices whose shortest distance from
the source is **final**, one vertex at a time, always taking the closest
remaining vertex (the *greedy choice*) and **relaxing** its outgoing edges. With
a binary-heap priority queue it runs in $O(E\log V)$. Its correctness rests
entirely on **non-negative edge weights**: that assumption is exactly what makes
a vertex's estimate optimal at the instant it is extracted — and, as the
counterexample showed, the algorithm can silently return wrong answers the
moment that assumption is broken.